<a href="https://colab.research.google.com/github/gunvarawork-cpu/code-repository/blob/main/LAB_4/LAB4_Geographic_Modeling_6606614623.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.0 MB/s eta 0:00:00


In [2]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='ee-gunvarawoon')

In [23]:
# =========================
# 1. Study Area
# =========================
roi = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM1_NAME', 'Kanchanaburi'))

# =========================
# 2. Load Data
# =========================
start = '2024-01-01'
end = '2024-05-31'

# Heat (LST)
lst = ee.ImageCollection('MODIS/061/MOD11A1') \
    .filterDate(start, end) \
    .select('LST_Day_1km') \
    .median().clip(roi)

# Landsat
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterDate(start, end) \
    .filterBounds(roi) \
    .filter(ee.Filter.lt('CLOUD_COVER', 30)) \
    .median().clip(roi)

# NDVI (Fuel)
ndvi = l8.normalizedDifference(['SR_B5', 'SR_B4'])

# NBR (Dryness)
nbr = l8.normalizedDifference(['SR_B5', 'SR_B7'])

# Fire History
history = ee.ImageCollection('FIRMS') \
    .filterDate('2022-01-01', '2023-12-31') \
    .select('T21') \
    .count().clip(roi)

# =========================
# 3. Normalize
# =========================
lst_norm = lst.unitScale(13000, 16000).clamp(0,1)

fuel_norm = ndvi.unitScale(0, 0.8).clamp(0,1)

dryness_norm = nbr.unitScale(-0.5, 0.5) \
    .subtract(1).multiply(-1).clamp(0,1)

history_norm = history.unitScale(0, 5).clamp(0,1)

# =========================
# 4. Model
# =========================
risk = dryness_norm.multiply(0.30) \
    .add(lst_norm.multiply(0.25)) \
    .add(fuel_norm.multiply(0.25)) \
    .add(history_norm.multiply(0.20)) \
    .rename('Fire_Risk') \
    .clip(roi)

# =========================
# 5. Classification (3 classes)
# =========================
risk_class = ee.Image(1) \
    .where(risk.gt(0.3), 2) \
    .where(risk.gt(0.5), 3) \
    .clip(roi)

# =========================
# 6. Visualization
# =========================
Map = geemap.Map()
Map.centerObject(roi, 8)

# 🔥 Risk (continuous)
Map.addLayer(risk, {
    'min':0,
    'max':0.7,
    'palette':['blue','green','yellow','orange','red']
}, 'Wildfire Risk')

# 🔥 Risk Class (3 classes)
Map.addLayer(risk_class, {
    'min':1,
    'max':3,
    'palette':['green','orange','red']
}, 'Risk Class')

# Boundary
Map.addLayer(roi, {'color':'black'}, 'Boundary')

# =========================
# LEGEND: Risk (gradient explanation)
# =========================
legend_risk = {
    'Low Risk': '#0000FF',
    'Moderate Risk': '#FFFF00',
    'High Risk': '#FF0000'
}
Map.add_legend(title='Wildfire Risk (Gradient)', legend_dict=legend_risk)

# =========================
# LEGEND: Risk Class (ชัดเจน)
# =========================
legend_class = {
    'Low (< 0.3)': '#00FF00',
    'Medium (0.3 – 0.5)': '#FFA500',
    'High (> 0.5)': '#FF0000'
}
Map.add_legend(title='Risk Classification', legend_dict=legend_class)

# =========================
# SHOW MAP
# =========================
Map

Map(center=[14.580790829178332, 99.05365917589702], controls=(WidgetControl(options=['position', 'transparent_…

In [24]:
# =========================
# Sensitivity Analysis
# =========================

# +20% dryness
risk_plus = dryness_norm.multiply(0.36) \
    .add(lst_norm.multiply(0.23)) \
    .add(fuel_norm.multiply(0.23)) \
    .add(history_norm.multiply(0.18)) \
    .clip(roi)

# -20% dryness
risk_minus = dryness_norm.multiply(0.24) \
    .add(lst_norm.multiply(0.27)) \
    .add(fuel_norm.multiply(0.27)) \
    .add(history_norm.multiply(0.22)) \
    .clip(roi)

# Difference
diff = risk_plus.subtract(risk).abs()

# =========================
# Threshold (กำหนดชัด)
# =========================
threshold = 0.03

robust = diff.lt(threshold)
sensitive = diff.gte(threshold)

# =========================
# Visualization
# =========================
Map = geemap.Map()
Map.centerObject(roi, 8)

# 🔥 Baseline (ดูภาพรวม)
Map.addLayer(risk, {
    'min':0,
    'max':0.7,
    'palette':['white','orange']
}, 'Baseline Risk')

# 🔥 Sensitivity (overlay)
Map.addLayer(robust.updateMask(robust), {
    'palette':['green']
}, 'Robust Area')

Map.addLayer(sensitive.updateMask(sensitive), {
    'palette':['red']
}, 'Sensitive Area')

# Boundary
Map.addLayer(roi, {'color':'black'}, 'Boundary')

# =========================
# LEGEND (ตรง threshold จริง)
# =========================
legend_sens = {
    'Robust (Δ < 0.03)': '#00FF00',
    'Sensitive (Δ ≥ 0.03)': '#FF0000'
}
Map.add_legend(title='Model Sensitivity', legend_dict=legend_sens)

# =========================
# SHOW MAP
# =========================
Map

Map(center=[14.580790829178332, 99.05365917589702], controls=(WidgetControl(options=['position', 'transparent_…

HTML(value="<html>\n<body>\n  <div class='my-legend'>\n  <div class='legend-title'>Model Sensitivity</div>\n  …

In [25]:
# =========================
# Validation: Burned Area
# =========================

# โหลด MODIS Burned Area 2024
burn = ee.ImageCollection('MODIS/061/MCD64A1') \
    .filterDate('2024-01-01', '2024-12-31') \
    .select('BurnDate') \
    .max() \
    .clip(roi)

# burned pixel = >0
burn_mask = burn.gt(0)

# High risk area (class 2-3)
high_risk = risk_class.gte(2)

# Pixel ที่ "ทำนายถูก" (intersection)
correct = burn_mask.And(high_risk)

# =========================
# Accuracy Calculation
# =========================

# burned จริงทั้งหมด
total_burned = burn_mask.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=500,
    maxPixels=1e13
).get('BurnDate').getInfo()

# burned ที่ model จับได้
correct_burned = correct.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=500,
    maxPixels=1e13
).get('BurnDate').getInfo()

# accuracy %
accuracy = (correct_burned / total_burned) * 100

print("Total burned pixels:", total_burned)
print("Correct predicted:", correct_burned)
print("Accuracy (%):", accuracy)

# =========================
# Visualization
# =========================
Map = geemap.Map()
Map.centerObject(roi, 8)

# 🔥 Risk (background)
Map.addLayer(risk, {
    'min':0,
    'max':0.7,
    'palette':['white','yellow','orange','red']
}, 'Wildfire Risk')

# 🔥 Burned Area (ของจริง)
Map.addLayer(burn_mask.selfMask(), {
    'palette':['black']
}, 'Observed Burned Area')

# 🔥 Correct Prediction (ซ้อนบน)
Map.addLayer(correct.selfMask(), {
    'palette':['blue']
}, 'Correct Prediction')

# Boundary
Map.addLayer(roi, {'color':'black'}, 'Boundary')

# =========================
# LEGEND (ตรง logic จริง)
# =========================
legend_val = {
    'Observed Burned Area': '#000000',
    'Correct Prediction (Hit)': '#0000FF'
}
Map.add_legend(title='Model Validation', legend_dict=legend_val)

# =========================
# SHOW MAP
# =========================
Map

Total burned pixels: 4600.12156862745
Correct predicted: 3148.5803921568627
Accuracy (%): 68.44559095198679


Map(center=[14.580790829178332, 99.05365917589702], controls=(WidgetControl(options=['position', 'transparent_…